<a href="https://colab.research.google.com/github/sathishweb997/ai-mentor-portfolio-sathishlutukurthi/blob/main/Day6_B_PlacementProcessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [ ]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

In [ ]:
with open('./sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] Ravi Kumar — 6 skills
  [2] Sneha Reddy — 6 skills
  [3] Arun Pillai — 8 skills
  [4] Priya Nair — 5 skills
  [5] Karthik Sharma — 5 skills

5/5 succeeded, 0 failed


In [ ]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Empty input: NameError: name 'extract_resume' is not defined
Whitespace input: NameError: name 'extract_resume' is not defined
Garbage input: NameError: name 'extract_resume' is not defined


In [1]:
from pydantic import BaseModel
from typing import List, Optional

class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [3]:
import requests
from bs4 import BeautifulSoup
import pathlib, json

def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text. Returns None on block / failure."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:max_chars]
    except Exception as e:
        print(f'  Scrape failed for {url}: {e}')
        return None

# Test on one URL
test_url = 'https://www.amazon.jobs/en-gb/jobs/10429173/engagement-manager-asean-professional-services'
text = fetch_jd(test_url)
if text:
    print(f'Got {len(text)} chars')
    print(text[:300])
else:
    print('Scrape blocked. Will use cached set.')

Got 6000 chars
Engagement Manager, ASEAN Professional Services - Job ID: 10429173 | Amazon.jobs
Skip to main content
×
Home
Teams
Locations
Job Categories
My career
My applications
My profile
Account security
Settings
Sign out
Resources
Disability accommodations
Benefits
Inclusive experiences
Interview tips
Leader


In [5]:
from pydantic import BaseModel
from typing import List, Optional
import requests
from bs4 import BeautifulSoup
from google import genai
import os
import getpass

# Initialize the genai client
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text. Returns None on block / failure."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:max_chars]
    except Exception as e:
        print(f'  Scrape failed for {url}: {e}')
        return None

def normalise_jd(text: str) -> JD:
    """Send JD text to Gemini, get structured JD JSON back."""
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Extract a JD JSON from this text:\n\n{text}',
        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )
    return JD.model_validate_json(resp.text)

# Define and fetch text within this cell for self-containment
test_url = 'https://www.amazon.jobs/en-gb/jobs/10429173/engagement-manager-asean-professional-services'
text = fetch_jd(test_url)

# Test on one JD text
if text:
    jd = normalise_jd(text)
    print(jd.model_dump_json(indent=2))
else:
    print('Failed to fetch JD text, cannot proceed with normalisation.')

Gemini API key: ··········
{
  "company": "Amazon Web Services",
  "role": "Engagement Manager, ASEAN Professional Services",
  "must_have_skills": [
    "Experience in cloud computing",
    "Experience in project management of technical programs, supporting Fortune 500 companies across multiple industries",
    "Bachelor's degree in Computer Science, Engineering, related field, or equivalent experience",
    "7+ years Project Management hands-on experience in managing and delivering enterprise level IT projects",
    "7+ years Program Management /Engagement Management experience leading other project managers to deliver a program with multiple and concurrent projects"
  ],
  "nice_to_have_skills": [
    "Project Management Professional (PMP)",
    "AWS Certified Solutions Architect - Associate",
    "Strong understanding of AWS services, architectures, and best practices",
    "Experience applying AWS frameworks like Well-Architected and Cloud Adoption Framework",
    "Proven ability 

In [6]:
import json, pathlib

URLS = [
    # Paste your 5 assigned URLs here
    'https://www.amazon.jobs/en-gb/jobs/10429173/engagement-manager-asean-professional-services',
    'https://www.amazon.jobs/en-gb/jobs/10423717/manager-site-merchandizing-rbs-retail-efficiency',
    'https://www.amazon.jobs/en-gb/jobs/10415014/finance-program-manager-aws-collections-team',
]

CACHE = pathlib.Path('./jds_cached.jsonl')
USE_CACHE = False   # set True if scraping is blocked

jds = []

if USE_CACHE and CACHE.exists():
    print(f'Using cached JDs from {CACHE}')
    for line in CACHE.read_text().splitlines():
        jds.append(JD.model_validate_json(line))
else:
    for url in URLS:
        text = fetch_jd(url)
        if text is None:
            continue
        try:
            jd = normalise_jd(text)
            jds.append(jd)
            print(f'  ✓ {jd.company} — {jd.role}')
        except Exception as e:
            print(f'  ✗ {url}: {e}')

print(f'\nProcessed {len(jds)} JDs')

# Inspect first 3
for jd in jds[:3]:
    print(f'\n{jd.company} - {jd.role}')
    print(f'  Must: {jd.must_have_skills}')
    print(f'  Nice: {jd.nice_to_have_skills}')
    print(f'  CGPA: {jd.min_cgpa}, LPA: {jd.package_lpa}')

  ✓ Amazon Web Services (AWS) — Engagement Manager, ASEAN Professional Services
  ✓ Amazon — Manager, Site Merchandizing , RBS Retail Efficiency
  ✓ Amazon — Finance Program Manager , AWS Collections Team

Processed 3 JDs

Amazon Web Services (AWS) - Engagement Manager, ASEAN Professional Services
  Must: ['Cloud computing', 'Project Management (7+ years in enterprise-level IT projects)', 'Program Management / Engagement Management (7+ years, leading multiple concurrent projects)', 'Knowledge of AWS services, architectures, and best practices', 'Technical project leadership', 'Bridging business requirements with technical solutions', 'Technical documentation']
  Nice: ['Project Management Professional (PMP) certification', 'AWS Certified Solutions Architect - Associate', 'Experience applying AWS frameworks (Well-Architected, Cloud Adoption Framework)', 'Ability to establish technical credibility with engineering teams and senior technical decision-makers']
  CGPA: None, LPA: None

Amaz

In [7]:
OUT = pathlib.Path('./jds.jsonl')
OUT.parent.mkdir(exist_ok=True)
with open(OUT, 'w') as f:
    for jd in jds:
        f.write(jd.model_dump_json() + '\n')
print(f'Wrote {len(jds)} JDs to {OUT}')

# Verify the file
with open(OUT) as f:
    for line in f:
        d = json.loads(line)
        print(f'  {d["company"]:20} | {d["role"]:30} | {len(d["must_have_skills"])} must-haves')

Wrote 3 JDs to jds.jsonl
  Amazon Web Services (AWS) | Engagement Manager, ASEAN Professional Services | 7 must-haves
  Amazon               | Manager, Site Merchandizing , RBS Retail Efficiency | 6 must-haves
  Amazon               | Finance Program Manager , AWS Collections Team | 9 must-haves
